# VAE hyperparameter search for cat faces - continued training (+40 epochs)

Picks up every run from `01_vae_4_hyperparam_search.ipynb` (listed in `hyperparam_search_summary.csv`), resumes each from its saved checkpoint, and trains for an additional 40 epochs (40 -> 80 total). Each continuation is written to its own new run directory so the original results are preserved.

## Imports

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from scripts import (
    DataFixedConfig,
    TrainFixedConfig,
    VAEFixedConfig,
    build_dataloaders,
    build_vae,
    compute_fid,
    count_parameters,
    output_paths,
    prepare_data,
    report_fid,
    resolve_device,
    run_name,
    save_history,
    save_history_plot,
    save_json,
    save_sample_grid,
    train_vae,
    update_total_stats,
    vae_sample_iterator,
)

## Parameters

In [ ]:
# Fixed config must match 01_vae_4_hyperparam_search.ipynb exactly, since these
# values aren't stored in each run's run.json (only the grid params are) and
# the model architecture/data pipeline must match the saved checkpoints.

EXPERIMENT_NAME = "vae"

# --- Data (must match the original sweep) -------------------------------
DATA_DIR = "../data/cats-faces"
OUTPUT_DIR = "../reports/runs"
CACHE_DIR = "../.cache/fid"
IMAGE_SIZE = 64
GRAYSCALE = False
POSTERIZE_BITS = 6

# --- Model (fixed, must match the original sweep) ------------------------
HIDDEN_DIM = 512
BASE_CHANNELS = 96
AMP = True

# --- Continuation ----------------------------------------------------------
ADDITIONAL_EPOCHS = 40
SUMMARY_PATH = Path(OUTPUT_DIR) / EXPERIMENT_NAME / "hyperparam_search_summary.csv"

# --- Early stopping (restarts fresh for the continuation) -----------------
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MIN_DELTA = 0.5
EARLY_STOPPING_RESTORE_BEST = True

# --- Checkpointing -----------------------------------------------------------
SAVE_CHECKPOINT = True

# --- Logging / sampling -----------------------------------------------------
FID_REAL_SAMPLES = 5000
FID_FAKE_SAMPLES = 5000
DEVICE = "cuda"
PROGRESS_BACKEND = "terminal"

data_fixed = DataFixedConfig(
    data_dir=DATA_DIR,
    cache_dir=CACHE_DIR,
    output_dir=OUTPUT_DIR,
    image_size=IMAGE_SIZE,
    channels=1 if GRAYSCALE else 3,
    grayscale=GRAYSCALE,
    posterize_bits=POSTERIZE_BITS,
)
model_fixed = VAEFixedConfig(
    base_channels=BASE_CHANNELS,
    hidden_dim=HIDDEN_DIM,
    amp=AMP,
)
train_fixed = TrainFixedConfig(
    device=DEVICE,
    progress_backend=PROGRESS_BACKEND,
    fid_real_samples=FID_REAL_SAMPLES,
    fid_fake_samples=FID_FAKE_SAMPLES,
    early_stopping=EARLY_STOPPING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    early_stopping_restore_best=EARLY_STOPPING_RESTORE_BEST,
)

summary_df = pd.read_csv(SUMMARY_PATH)
print(f"Loaded {len(summary_df)} prior run(s) from {SUMMARY_PATH}")
summary_df

## Data

All prior runs share the same data configuration, so it is loaded from the first run's `run.json` and prepared once for every continuation.

In [ ]:
first_run_dir = summary_df.iloc[0]["run_dir"]
first_config = json.loads(
    (Path(OUTPUT_DIR) / EXPERIMENT_NAME / first_run_dir / "configs" / "run.json").read_text()
)
data_params = first_config["data"]

torch.manual_seed(data_params["seed"])
np.random.seed(data_params["seed"])

prepared = prepare_data(
    data_fixed,
    train_fraction=data_params["train_fraction"],
    validation_fraction=data_params["validation_fraction"],
    augment_flip=data_params["augment_flip"],
    seed=data_params["seed"],
)
print(f"Train images: {len(prepared.train_dataset)} | Validation: {len(prepared.validation_dataset)}")

train_loader, validation_loader = build_dataloaders(
    prepared,
    data_fixed,
    batch_size=first_config["train"]["batch_size"],
    seed=data_params["seed"],
)

device = resolve_device(train_fixed.device)
print(f"Using device: {device}")

## Continue training

For each prior run, resumes from its checkpoint and trains for `ADDITIONAL_EPOCHS` more epochs, saving artifacts to a new run directory.

In [ ]:
results = []

for i, row in summary_df.iterrows():
    prev_run_dir = row["run_dir"]
    prev_root = Path(OUTPUT_DIR) / EXPERIMENT_NAME / prev_run_dir
    prev_config = json.loads((prev_root / "configs" / "run.json").read_text())
    prev_checkpoint = prev_root / "checkpoints" / "vae_final.pt"

    model_params = dict(prev_config["model"])
    data_params = prev_config["data"]
    new_epochs = prev_config["train"]["epochs"] + ADDITIONAL_EPOCHS

    print(
        f"\n=== Run {i + 1}/{len(summary_df)}: latent_dim={model_params['latent_dim']} "
        f"beta={model_params['beta']} learning_rate={model_params['learning_rate']} "
        f"({prev_config['train']['epochs']} -> {new_epochs} epochs) ==="
    )

    run = {
        "experiment": EXPERIMENT_NAME,
        "kind": "vae",
        "data": data_params,
        "train": {**prev_config["train"], "epochs": new_epochs},
        "model": model_params,
    }

    torch.manual_seed(data_params["seed"])
    np.random.seed(data_params["seed"])

    run_dir = run_name(run["kind"], run)
    paths = output_paths(EXPERIMENT_NAME, OUTPUT_DIR, run_dir)
    save_json(paths["configs"] / "run.json", run)
    save_json(paths["configs"] / "resumed_from.json", {"run_dir": prev_run_dir})

    model = build_vae(model_params, model_fixed, data_fixed)
    print(f"VAE parameters: {count_parameters(model):,}")

    training_result = train_vae(
        model,
        train_loader,
        validation_loader,
        model_params=model_params,
        train_params=run["train"],
        train_fixed=train_fixed,
        fixed_params=model_fixed,
        sample_dir=paths["samples"],
        checkpoint_path=paths["checkpoints"] / "vae_final.pt" if SAVE_CHECKPOINT else None,
        resume_from=prev_checkpoint,
    )

    history = training_result["history"]
    save_history(history, paths["metrics"] / "history.csv")
    save_history_plot(
        history,
        paths["figures"] / "loss_curves.png",
        metrics=["loss", "validation_loss", "reconstruction", "kl"],
        title="VAE Loss Curves",
    )
    elapsed = training_result["elapsed_seconds"]
    print(f"Trained in {int(elapsed // 60):02d}:{int(elapsed % 60):02d} on {training_result['device']}.")
    update_total_stats(OUTPUT_DIR, run["kind"], run_dir, EXPERIMENT_NAME, elapsed, training_result["device"])

    with torch.no_grad():
        z = torch.randn(64, model.latent_dim, device=device)
        save_sample_grid(model.decode(z), paths["figures"] / "samples_final.png", nrow=8)

    sampler = vae_sample_iterator(
        model,
        num_samples=train_fixed.fid_fake_samples,
        batch_size=run["train"]["batch_size"],
        device=device,
        seed=data_params["seed"],
    )
    fid_start = time.perf_counter()
    fid_score = compute_fid(
        sampler,
        data_fixed,
        num_real_samples=train_fixed.fid_real_samples,
        seed=data_params["seed"],
        device=device,
    )
    fid_elapsed = time.perf_counter() - fid_start
    report_fid(fid_score, elapsed_seconds=fid_elapsed)
    save_json(paths["metrics"] / "fid.json", {"fid": fid_score, "elapsed_seconds": fid_elapsed})

    last = history[-1]
    results.append({
        "run_dir": run_dir,
        "previous_run_dir": prev_run_dir,
        "latent_dim": model_params["latent_dim"],
        "beta": model_params["beta"],
        "learning_rate": model_params["learning_rate"],
        "epochs_run": len(history),
        "fid": fid_score,
        "previous_fid": row["fid"],
        "val_loss": last["validation_loss"],
        "val_reconstruction": last["validation_reconstruction"],
        "val_kl": last["validation_kl"],
        "elapsed_seconds": elapsed,
    })

    del model
    torch.cuda.empty_cache()

## Results

In [ ]:
results_df = pd.DataFrame(results)
results_df["fid_delta"] = results_df["fid"] - results_df["previous_fid"]
results_df = results_df.sort_values("fid").reset_index(drop=True)
results_df.to_csv(Path(OUTPUT_DIR) / EXPERIMENT_NAME / "hyperparam_search_summary_02.csv", index=False)
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(results_df["previous_fid"], results_df["fid"], alpha=0.7)
lims = [
    min(results_df["previous_fid"].min(), results_df["fid"].min()) - 2,
    max(results_df["previous_fid"].max(), results_df["fid"].max()) + 2,
]
axes[0].plot(lims, lims, color="gray", linestyle="--", label="no change")
axes[0].set_xlabel("FID after 40 epochs")
axes[0].set_ylabel("FID after 80 epochs")
axes[0].set_title("FID before vs. after continuation")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(results_df["run_dir"], results_df["fid_delta"])
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set_ylabel("FID delta (80ep - 40ep)")
axes[1].set_title("Change in FID per run")
axes[1].tick_params(axis="x", rotation=90, labelsize=7)
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, param in zip(axes, ["latent_dim", "beta", "learning_rate"]):
    means = results_df.groupby(param)["fid"].mean()
    ax.scatter(results_df[param], results_df["fid"], alpha=0.5, label="run")
    ax.plot(means.index, means.values, marker="o", color="black", label="mean")
    ax.set_xlabel(param)
    ax.set_ylabel("FID")
    ax.set_title(f"FID vs {param} (after 80 epochs)")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()